# PyTorch Training Skeleton: Quick Refresher

**Guiding question:** What is the smallest reusable PyTorch training skeleton worth remembering?

This is the shortcut route for readers who have trained neural networks before and need the whole contract on one screen:

```text
examples
-> Dataset
-> DataLoader
-> nn.Module.forward
-> logits
-> loss
-> backward
-> optimizer step
-> inference
```

**How this notebook fits the track**

- **New to PyTorch:** follow Parts 00–04 in order; they prove each responsibility separately.
- **Returning to PyTorch:** use this notebook as a stand-in for Parts 00–04, then continue to the sequence and language-model chapters.
- **The gate:** the stand-in is sufficient only if the closing checks make sense without guesswork. The handoff table points back to the exact chapter for anything rusty.

This notebook intentionally omits devices, validation, schedulers, mixed precision, checkpoints, and distributed training. Those are lifecycle layers around this irreducible core, not replacements for it.

## 1 - A loader turns examples into batches

Each example has:

- two floating-point input features;
- one integer class target.

`TensorDataset` binds row `i` of the inputs to row `i` of the targets. `DataLoader` chooses how examples are delivered:

- `batch_size=4` groups four examples per update;
- `shuffle=True` changes example order between epochs;
- shuffling does not change feature columns or break input-target pairs.

For classification with `CrossEntropyLoss`, targets must be integer class IDs with dtype `torch.long`. They are not one-hot vectors.

**Shape contract**

```text
inputs   [examples, features]
targets  [examples]
batch    [batch size, features], [batch size]
```

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
torch.manual_seed(SEED)

inputs = torch.tensor(
    [
        [-1.2, -0.8],
        [-1.0, -1.3],
        [-0.7, -0.9],
        [-1.3, -1.1],
        [0.8, 1.1],
        [1.2, 0.7],
        [0.9, 1.4],
        [1.3, 1.0],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)

dataset = TensorDataset(inputs, targets)
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

batch_inputs, batch_targets = next(iter(loader))
print("Full data:", tuple(inputs.shape), tuple(targets.shape))
print("One batch:", tuple(batch_inputs.shape), tuple(batch_targets.shape))
assert batch_inputs.shape == (4, 2)
assert batch_targets.dtype == torch.long

## 2 - A model class owns trainable layers and prediction

Every ordinary PyTorch network inherits from `nn.Module`.

`__init__` declares layers and gives trainable state a home. Calling `super().__init__()` lets PyTorch register child modules and parameters so operations such as `model.parameters()`, saving, and device movement can find them.

`forward` describes the prediction path. You normally call `model(inputs)`, not `model.forward(inputs)`, because `nn.Module.__call__` wraps `forward` with PyTorch's hook and framework behavior.

The final layer emits two **logits**, one score per class. Do not add softmax before `CrossEntropyLoss`; the loss applies the numerically stable operation it needs.

PyTorch requires `forward`. It does not require `fit`. Training orchestration can remain a separate function so the same model can be paired with different data, losses, and optimizers.

In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, inputs):
        return self.network(inputs)


model = TinyClassifier(input_size=2, hidden_size=8, output_size=2)
logits = model(batch_inputs)

print("Batch inputs:", tuple(batch_inputs.shape))
print("Batch logits:", tuple(logits.shape))
print("Trainable tensors:", [name for name, _ in model.named_parameters()])
assert logits.shape == (len(batch_inputs), 2)

## 3 - Loss measures; backward reports; the optimizer acts

The reusable loop has five ordered operations:

```text
optimizer.zero_grad()
logits = model(batch_inputs)
loss = loss_function(logits, batch_targets)
loss.backward()
optimizer.step()
```

Their jobs are different:

1. `zero_grad` clears gradients left by earlier updates. Gradients accumulate by default.
2. The forward call computes predictions using the current weights.
3. The loss compares the whole batch with its targets and returns one scalar.
4. `backward` fills parameter `.grad` buffers; it does not change weights.
5. `step` applies the optimizer's update rule and changes weights.

`CrossEntropyLoss` uses the mean example loss by default. To report a correct epoch average when the last batch may be smaller, multiply each batch mean by its number of examples, add those totals, then divide by the dataset size.

The optimizer receives `model.parameters()` because those are the tensors it must update. Adam also keeps its own moving state; that state belongs to training, not to `forward`.

In [ ]:
def fit(model, dataloader, loss_function, optimizer, epochs):
    history = []

    for _ in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_inputs, batch_targets in dataloader:
            optimizer.zero_grad()
            logits = model(batch_inputs)
            loss = loss_function(logits, batch_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_inputs)

        average_loss = total_loss / len(dataloader.dataset)
        history.append(average_loss)

    return history

## 4 - Wire the independent pieces together

The model does not choose its own objective or update rule:

- the task chooses the loss;
- the experiment chooses the optimizer and learning rate;
- `fit` coordinates them.

That separation is why the same skeleton scales from this classifier to CBOW and Transformers.

After training, `model.eval()` selects evaluation behavior for layers such as dropout. `torch.inference_mode()` disables gradient bookkeeping when no update will follow. The forward computation still runs; only training-only graph construction disappears.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
history = fit(
    model,
    loader,
    loss_function,
    optimizer,
    epochs=80,
)

model.eval()
with torch.inference_mode():
    final_logits = model(inputs)
    predictions = final_logits.argmax(dim=1)

accuracy = (predictions == targets).float().mean().item()
print(f"Average loss: {history[0]:.4f} -> {history[-1]:.4f}")
print(f"Training accuracy: {accuracy:.0%}")
print("Predictions:", predictions.tolist())

assert history[-1] < history[0]
assert accuracy == 1.0
assert not final_logits.requires_grad

## 5 - The same skeleton becomes CBOW

The [CBOW capstone](../../03-language-model-foundations/02-tokenization-and-embeddings/02C-context-windows-to-static-embeddings.ipynb) changes the data and forward path, not the training contract:

| Refresher classifier | CBOW |
|---|---|
| Feature batch `[B, 2]` | Context token IDs `[B, C]` |
| Linear feature projection | Embedding lookup `[B, C, D]` |
| Hidden representation | Mean context representation `[B, D]` |
| Two class logits `[B, 2]` | Vocabulary logits `[B, V]` |
| Integer class target `[B]` | Integer center-token target `[B]` |
| Cross-entropy | Cross-entropy |
| `zero_grad -> forward -> loss -> backward -> step` | The same five operations |

That makes CBOW the first compact language-shaped module in the track. A Transformer makes `forward` much richer, but it still plugs into this loop.

## Stand-in readiness check

You are ready to use this notebook as the shortcut for Parts 00–04 if you can explain all five statements:

1. `__init__` registers trainable layers; `forward` defines prediction.
2. A `DataLoader` changes batch delivery, not model mathematics.
3. `backward()` fills gradients; only `optimizer.step()` updates weights.
4. Epoch loss must account for different batch sizes.
5. Evaluation uses `model.eval()` and `torch.inference_mode()`.

If one statement is fuzzy, take the targeted route rather than replaying everything:

| Gap | Resume here |
|---|---|
| Tensors, shapes, dtypes, broadcasting, or autograd | [Part 02](../02-tensors-and-autograd/02-tensors-and-autograd.ipynb) |
| `nn.Module`, parameter ownership, loss, or optimizer order | [Part 03](../03-models-and-training-loops/03-models-and-training-loops.ipynb) |
| `Dataset`, `DataLoader`, checkpoints, or optimizer state | [Part 04](../04-data-checkpoints-and-hyperparameters/04-data-checkpoints-and-hyperparameters.ipynb) |
| Validation, selection, resume, AMP, or inference lifecycle | [Part 05](../05-modern-training-lifecycle/05-modern-training-lifecycle.ipynb) |

**Next on the shortcut route:** continue to [Sequence Memory](../../03-language-model-foundations/01-sequence-memory/README.md), or go directly to [Tokenization and Embeddings](../../03-language-model-foundations/02-tokenization-and-embeddings/README.md) when recurrence is not your current goal.

## Copy/paste memory card

This is the same executable contract with the teaching output removed:

```python
class Model(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, inputs):
        return self.network(inputs)


def fit(model, dataloader, loss_function, optimizer, epochs):
    history = []
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_function(predictions, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(inputs)
        history.append(total_loss / len(dataloader.dataset))
    return history


dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
model = Model(input_size, hidden_size, output_size)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = fit(model, dataloader, loss_function, optimizer, epochs=10)

model.eval()
with torch.inference_mode():
    predictions = model(new_inputs)
```

### Coverage ledger

**Built and measured:** `TensorDataset`, `DataLoader`, `nn.Module`, `__init__`, `forward`, logits, cross-entropy, Adam, a reusable `fit`, correctly aggregated epoch loss, evaluation, and inference.

**Explained or linked:** parameter registration, optimizer state, classification-to-CBOW shape mapping, and the full-route handoff.

**Named and out of scope:** device placement, validation, checkpoints, schedulers, mixed precision, clipping, and distributed training. Part 05 adds those only after this loop is routine.